<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/dpr_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install transformers "datasets<3" sentencepiece accelerate faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
import re
import torch
import faiss
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    DPRContextEncoder,
    DPRContextEncoderTokenizer
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to {device}")

torch.manual_seed(0)

Device set to cpu


# NQ dataset

In [3]:
nq = load_dataset("sentence-transformers/natural-questions", split="train[:500]")
dataset_name = "nq"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/100231 [00:00<?, ? examples/s]

# Helper functions

In [4]:
def get_question(example):
  return example["query"]

def get_answer(example):
  answer = example["answer"]
  if isinstance(answer, list):
    answer = answer[0]
  return answer

In [5]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Token-level F1

In [ ]:
def qa_f1(prediction, gold):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)

    common = set(pred_tokens) & set(gold_tokens)
    num_same = sum(
        min(pred_tokens.count(tok), gold_tokens.count(tok))
        for tok in common
    )

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

# Build Passage Corpus

In [6]:
def build_passage_corpus(dataset, answer_field_fn):
  passages = []
  for i in range(len(dataset)):
    passages.append(answer_field_fn(dataset[i]))
  passages = list(dict.fromkeys(passages))
  return passages

passages = build_passage_corpus(nq, get_answer)
print(f"Number of passages: {len(passages)}")
print(passages[0])

Number of passages: 497
Richmond Football Club Richmond began 2017 with 5 straight wins, a feat it had not achieved since 1995. A series of close losses hampered the Tigers throughout the middle of the season, including a 5-point loss to the Western Bulldogs, 2-point loss to Fremantle, and a 3-point loss to the Giants. Richmond ended the season strongly with convincing victories over Fremantle and St Kilda in the final two rounds, elevating the club to 3rd on the ladder. Richmond's first final of the season against the Cats at the MCG attracted a record qualifying final crowd of 95,028; the Tigers won by 51 points. Having advanced to the first preliminary finals for the first time since 2001, Richmond defeated Greater Western Sydney by 36 points in front of a crowd of 94,258 to progress to the Grand Final against Adelaide, their first Grand Final appearance since 1982. The attendance was 100,021, the largest crowd to a grand final since 1986. The Crows led at quarter time and led by as

# Load DPR models

In [7]:
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = question_encoder.to(device)


context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = context_encoder.to(device)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


# Question Encoding

In [8]:
def encode_question(question):
  inputs = question_tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}
  with torch.no_grad():
    outputs = question_encoder(**inputs).pooler_output
  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Passage Encoding

In [9]:
def encode_passage(passage):
  inputs = context_tokenizer(passage, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = context_encoder(**inputs).pooler_output

  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Build FAISS index

In [10]:
def build_faiss_index(passages):
  passages_embeddings = np.stack([encode_passage(passage) for passage in passages]).astype(np.float32)
  faiss.normalize_L2(passages_embeddings)
  index = faiss.IndexFlatIP(passages_embeddings.shape[1])
  index.add(passages_embeddings)
  return index

index = build_faiss_index(passages)

# Retrieval function

In [11]:
def retrieval_top_k(question, index, passages, k = 5):
  question_embedding = encode_question(question)
  question_embedding = question_embedding.reshape(1, -1)
  faiss.normalize_L2(question_embedding)
  distances, indices = index.search(question_embedding, k)
  relevant_passages = [passages[i] for i in indices[0]]
  return relevant_passages, distances[0]

# Prediction rule

In [12]:
def dpr_prediction_rule(question, index, passages, k = 5):
  relevant_passages, distances = retrieval_top_k(question, index, passages, k)
  prediction = relevant_passages[0]
  return prediction, relevant_passages

# Smoke Test

In [13]:
question = get_question(nq[0])
answer = get_answer(nq[0])

prediction, relevant_passages = dpr_prediction_rule(question, index, passages)

print("Question: ", question)
print("Answer: ", answer)
print("Prediction: ", prediction)
print("Relevant passages: ", relevant_passages)

Question:  when did richmond last play in a preliminary final
Answer:  Richmond Football Club Richmond began 2017 with 5 straight wins, a feat it had not achieved since 1995. A series of close losses hampered the Tigers throughout the middle of the season, including a 5-point loss to the Western Bulldogs, 2-point loss to Fremantle, and a 3-point loss to the Giants. Richmond ended the season strongly with convincing victories over Fremantle and St Kilda in the final two rounds, elevating the club to 3rd on the ladder. Richmond's first final of the season against the Cats at the MCG attracted a record qualifying final crowd of 95,028; the Tigers won by 51 points. Having advanced to the first preliminary finals for the first time since 2001, Richmond defeated Greater Western Sydney by 36 points in front of a crowd of 94,258 to progress to the Grand Final against Adelaide, their first Grand Final appearance since 1982. The attendance was 100,021, the largest crowd to a grand final since 19

# Evaluation

In [14]:
results = []

for i in range(50):
  question = get_question(nq[i])
  answer = get_answer(nq[i])
  prediction, relevant_passages = dpr_prediction_rule(question, index, passages)
  f1 = qa_f1(prediction, answer)

  results.append({
      "dataset": dataset_name,
      "question_number": i,
      "question": question,
      "answer": answer,
      "prediction": prediction,
      "f1": f1,
      "relevant_passages": relevant_passages
  })

results_df = pd.DataFrame(results)
results_df.head()

,dataset,question_number,question,answer,prediction,em,relevant_passages
0,nq,0,when did richmond last play in a preliminary f...,Richmond Football Club Richmond began 2017 wit...,Richmond Football Club Richmond began 2017 wit...,1,[Richmond Football Club Richmond began 2017 wi...
1,nq,1,who sang what in the world's come over you,"Jack Scott (singer) At the beginning of 1960, ...","The Lion Sleeps Tonight ""The Lion Sleeps Tonig...",0,"[The Lion Sleeps Tonight ""The Lion Sleeps Toni..."
2,nq,2,who produces the most wool in the world,Wool Global wool production is about 2 million...,Wool Global wool production is about 2 million...,1,[Wool Global wool production is about 2 millio...
3,nq,3,where does alaska the last frontier take place,Alaska: The Last Frontier Alaska: The Last Fro...,The empire on which the sun never sets In the ...,0,[The empire on which the sun never sets In the...
4,nq,4,a day to remember all i want cameos,All I Want (A Day to Remember song) The music ...,"The Big Sick During the credits, photos are sh...",0,"[The Big Sick During the credits, photos are s..."


In [15]:
final_f1 = results_df["f1"].mean()*100
print("DPR F1: ", final_f1)
results_df.to_csv("dpr_results.csv", index=False)
print("Saved dpr_results.csv")

DPR EM:  42.0
Saved dpr_results.csv
